# Pokémon TCG — Deck Analysis & RL Data Prep

This notebook has **two jobs**:

1. **Data exploration (for the team).** Walk through the prepared card database, split the
   cards into the categories the game actually distinguishes, and explain *why* those
   categories matter under the Pokémon TCG rules. Read this section top-to-bottom if you're
   new to the dataset.
2. **RL feature prep.** Turn the raw card database into clean, numeric, machine-ingestible
   tables that a reinforcement-learning agent can use to **(a) build a legal deck**,
   **(b) play games**, and **(c) reevaluate and refine its deck** across many games.

**Data sources**
- `data/EN_Card_Data.csv` — human-readable card database (one row *per attack/ability*, so
  cards with multiple moves span multiple rows). Good for exploration and effect text.
- `cg.api.all_card_data()` / `all_attack()` — the **engine's** canonical, already-typed view
  of the exact same 1,267 cards. This is the ground truth the agent sees at runtime, so we
  base the RL features on it and keep the CSV for the readable labels/effect text.

> Card IDs are shared across both sources, so we can freely join them on `Card ID` / `cardId`.

## 1. Data exploration — the card database

We load the CSV and take a first look. Because a card with two attacks and an ability
occupies **three rows**, the raw row count (~2,022) is larger than the number of distinct
cards (1,267). Keep that in mind: *rows are moves, `Card ID` is the card.*

**Column dictionary**

| Column | Meaning |
|---|---|
| `Card ID` | Stable numeric id — the key everything joins on |
| `Card Name` | Display name (e.g. `Mega Lucario ex`) |
| `Expansion` / `Collection No.` | Which set the card is from |
| `Stage (Pokémon)/Type (Energy and Trainer)` | The **super-type** — the most important column (see below) |
| `Rule` | Special rule overlay: `Pokémon ex`, `Mega Pokémon ex`, `ACE SPEC`, or `n/a` |
| `Category` | Mechanic tag: `Ancient`, `Future`, `Tera(Stellar)`, `Trainer's Pokémon (…)`, … |
| `Previous stage` | For evolutions, the card it evolves from |
| `HP`, `Type`, `Weakness`, `Resistance (Type)`, `Retreat` | Pokémon battle stats |
| `Move Name`, `Cost`, `Damage`, `Effect Explanation` | One attack/ability per row |

In [24]:
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

# The card database: one row per attack/ability, so cards with several moves span several rows.
df = pl.read_csv(Path("data/EN_Card_Data.csv"))

SUPERTYPE = "Stage (Pokémon)/Type (Energy and Trainer)"  # the column we lean on constantly

print(f"rows (moves): {df.height:>6}")
print(f"distinct cards: {df['Card ID'].n_unique():>4}")
print(f"columns: {df.columns}")

# The nine super-types every card falls into:
df.select(pl.col(SUPERTYPE)).unique()

rows (moves):   2022
distinct cards: 1267
columns: ['Card ID', 'Card Name', 'Expansion', 'Collection No.', 'Stage (Pokémon)/Type (Energy and Trainer)', 'Rule', 'Category', 'Previous stage', 'HP', 'Type', 'Weakness', 'Resistance (Type)', 'Retreat', 'Move Name', 'Cost', 'Damage', 'Effect Explanation']


Stage (Pokémon)/Type (Energy and Trainer)
str
"""Basic Energy"""
"""Basic Pokémon"""
"""Supporter"""
"""Pokémon Tool"""
"""Special Energy"""
"""Stage 2 Pokémon"""
"""Stadium"""
"""Item"""
"""Stage 1 Pokémon"""


### The three families every card belongs to

That one super-type column collapses into **three families**, which is exactly how the rules
treat cards. A legal 60-card deck is built entirely out of these:

| Family | Super-types in the data | What it does in the game |
|---|---|---|
| **Pokémon** | `Basic Pokémon`, `Stage 1 Pokémon`, `Stage 2 Pokémon` | Your board. They take the Active/Bench spots, hold Energy, attack, and are what you Knock Out to take Prize cards. |
| **Trainer** | `Item`, `Supporter`, `Stadium`, `Pokémon Tool` | Support cards you play from hand for effects (draw, search, switch, heal, disrupt). |
| **Energy** | `Basic Energy`, `Special Energy` | Attached to Pokémon to pay for attacks and retreat. |

> ⚠️ Note the data quirk: `Pokémon Tool` contains the word *"Pokémon"* but is a **Trainer**
> card, and Energy super-types contain *"Energy"*. So the cleanest way to split is on the
> **leading keyword** of the super-type, which we do below.

We first collapse the move-level rows down to **one row per card** (`cards`), which is the
right grain for counting how many cards of each kind exist.

In [25]:
# The CSV has no fully-duplicated rows...
assert df.is_duplicated().sum() == 0

# ...but it does repeat Card ID across a card's several moves. Collapse to one row per card:
# the card-level attributes are identical on every move row, so `first` is safe.
CARD_ATTRS = [SUPERTYPE, "Card Name", "Rule", "Category", "Previous stage",
              "HP", "Type", "Weakness", "Resistance (Type)", "Retreat", "Expansion"]

cards = (
    df.group_by("Card ID")
      .agg([pl.col(c).first() for c in CARD_ATTRS] + [pl.len().alias("n_moves")])
      .sort("Card ID")
)

# Assign the three families from the leading keyword of the super-type.
cards = cards.with_columns(
    pl.when(pl.col(SUPERTYPE).str.contains("Energy")).then(pl.lit("Energy"))
      .when(pl.col(SUPERTYPE).str.ends_with("Pokémon")).then(pl.lit("Pokémon"))  # Basic/Stage 1/Stage 2
      .otherwise(pl.lit("Trainer"))                                               # Item/Supporter/Stadium/Tool
      .alias("family")
)

cards.group_by("family").agg(pl.len().alias("n_cards")).sort("n_cards", descending=True)

family,n_cards
str,u32
"""Pokémon""",1056
"""Trainer""",191
"""Energy""",20


In [26]:
# One card-level frame per family. We'll drill into each below.
pokemon_db = cards.filter(pl.col("family") == "Pokémon")
trainer_db = cards.filter(pl.col("family") == "Trainer")
energy_db  = cards.filter(pl.col("family") == "Energy")

print(f"Pokémon: {pokemon_db.height:>4} cards")
print(f"Trainer: {trainer_db.height:>4} cards")
print(f"Energy : {energy_db.height:>4} cards")

Pokémon: 1056 cards
Trainer:  191 cards
Energy :   20 cards


### 1a. Pokémon — evolution stages

Pokémon come in an **evolution chain**, and the stage controls *how a card gets into play*:

- **Basic Pokémon** — can be played straight from your hand onto the Bench. Every deck
  **must** contain at least one Basic (you can't start a game without one in play).
- **Stage 1 Pokémon** — cannot be played directly; you must *evolve* it on top of the Basic
  named in its `Previous stage` (e.g. `Riolu → Lucario`).
- **Stage 2 Pokémon** — evolves on top of the matching Stage 1 (`Basic → Stage 1 → Stage 2`),
  so it takes longer to set up but is usually the strongest.

A Pokémon can only evolve on a **later turn** than it came into play (with a few card
exceptions), so evolution lines cost tempo — an important trade-off for deck building.

> `Pokémon Tool` was filtered out into `trainer_db` above, so `pokemon_db` is exactly the
> three battle stages.

In [27]:
# Stage breakdown
print(pokemon_db.group_by(SUPERTYPE).agg(pl.len().alias("n")).sort("n", descending=True))

# Trace an example evolution line via the `Previous stage` column: Riolu -> Lucario -> ...
line = pokemon_db.filter(
    pl.col("Card Name").str.contains("Lucario") | pl.col("Card Name").str.contains("Riolu")
).select(["Card ID", "Card Name", SUPERTYPE, "Previous stage", "HP"])
line

shape: (3, 2)
┌───────────────────────────────────────────┬─────┐
│ Stage (Pokémon)/Type (Energy and Trainer) ┆ n   │
│ ---                                       ┆ --- │
│ str                                       ┆ u32 │
╞═══════════════════════════════════════════╪═════╡
│ Basic Pokémon                             ┆ 595 │
│ Stage 1 Pokémon                           ┆ 345 │
│ Stage 2 Pokémon                           ┆ 116 │
└───────────────────────────────────────────┴─────┘


Card ID,Card Name,Stage (Pokémon)/Type (Energy and Trainer),Previous stage,HP
i64,str,str,str,str
333,"""Riolu""","""Basic Pokémon""","""n/a""","""70"""
677,"""Riolu""","""Basic Pokémon""","""n/a""","""80"""
678,"""Mega Lucario ex""","""Stage 1 Pokémon""","""Riolu""","""340"""
974,"""Riolu""","""Basic Pokémon""","""n/a""","""70"""


### 1b. Special Pokémon rule overlays

On top of the stage, a Pokémon can carry **rule overlays** that change how many Prize cards
the opponent takes when it's Knocked Out, or add global mechanics. These live in the `Rule`
and `Category` columns and are **decisive for strategy** — this is the single biggest lever
on how "swingy" a game is.

**Prize-card rule (from `Rule`)** — you win by taking all your Prize cards, so how many a
Pokémon *gives up* is its risk:

| `Rule` value | What it is | Prizes given on KO |
|---|---|---|
| `n/a` | Ordinary Pokémon | **1** |
| `Pokémon ex` | Powerful, high-HP attacker | **2** |
| `Mega Pokémon ex` | Mega Evolution ex | **3** |

So a `Mega ... ex` is a huge damage threat but hands the opponent 3 of their 6 Prizes if it
falls — half the game in one Knock Out.

**Global mechanic tags (from `Category`)** — these gate which support cards synergize:

- **`Ancient` / `Future`** — paradox Pokémon; several Trainer/Stadium cards only boost one group.
- **`Tera(Stellar)`** — **Tera** Pokémon take **no damage while on the Bench**, changing target priority.
- **`Trainer's Pokémon (…)`** — themed Pokémon (e.g. *Team Rocket's*, *N's*, *Ethan's*) that
  pair with same-owner Trainer cards and can require special Energy (e.g. `Team Rocket's Energy`).

`ACE SPEC` also appears in `Rule` for a few Trainer/Energy cards — see the deck-building rules
in Section 2 (you may run **at most one** ACE SPEC in a deck).

In [28]:
# Turn the Rule/Category text into explicit strategic flags + the Prize value of each Pokémon.
pokemon_tagged = pokemon_db.with_columns(
    is_ex          = pl.col("Rule").is_in(["Pokémon ex", "Mega Pokémon ex"]),
    is_mega_ex     = pl.col("Rule") == "Mega Pokémon ex",
    is_tera        = pl.col("Category").str.contains("Tera"),
    is_ancient     = pl.col("Category") == "Ancient",
    is_future      = pl.col("Category") == "Future",
    is_trainers_mon = pl.col("Category").str.contains("Trainer's"),
).with_columns(
    # Prizes the opponent takes when this Pokémon is Knocked Out.
    prizes_on_ko = pl.when(pl.col("is_mega_ex")).then(3)
                     .when(pl.col("is_ex")).then(2)
                     .otherwise(1)
)

print("Prize distribution:")
print(pokemon_tagged.group_by("prizes_on_ko").agg(pl.len().alias("n")).sort("prizes_on_ko"))
print("\nOverlay counts:")
print(pokemon_tagged.select(
    pl.col("^is_.*$").sum()
))

Prize distribution:
shape: (3, 2)
┌──────────────┬─────┐
│ prizes_on_ko ┆ n   │
│ ---          ┆ --- │
│ i32          ┆ u32 │
╞══════════════╪═════╡
│ 1            ┆ 905 │
│ 2            ┆ 121 │
│ 3            ┆ 30  │
└──────────────┴─────┘

Overlay counts:
shape: (1, 6)
┌───────┬────────────┬─────────┬────────────┬───────────┬─────────────────┐
│ is_ex ┆ is_mega_ex ┆ is_tera ┆ is_ancient ┆ is_future ┆ is_trainers_mon │
│ ---   ┆ ---        ┆ ---     ┆ ---        ┆ ---       ┆ ---             │
│ u32   ┆ u32        ┆ u32     ┆ u32        ┆ u32       ┆ u32             │
╞═══════╪════════════╪═════════╪════════════╪═══════════╪═════════════════╡
│ 151   ┆ 30         ┆ 32      ┆ 12         ┆ 8         ┆ 148             │
└───────┴────────────┴─────────┴────────────┴───────────┴─────────────────┘


### 1c. Trainer cards — the four sub-types

Trainer cards are played from hand for effects. The sub-type controls **how often** you can
play them, which is the whole balancing mechanism:

| Sub-type | Play limit per turn | Role |
|---|---|---|
| **Item** | Unlimited | Cheap utility: search, draw, switch, tools-of-the-trade. |
| **Supporter** | **1 per turn** | The powerful cards (big draw, search, disruption). The 1/turn cap is why draw engines are so contested. |
| **Stadium** | **1 in play, globally** | Stays on the field and affects **both** players until replaced. Only one Stadium exists at a time. |
| **Pokémon Tool** | Unlimited to attach, 1 per Pokémon | Attached to a Pokémon to buff it (extra HP, effects). Stays until the Pokémon leaves play. |

Some Trainers are also **`ACE SPEC`** (see `Rule`) — extremely strong, but you may include
**only one ACE SPEC card total** across your whole deck.

In [29]:
# Trainer sub-types, with how many are ACE SPEC in each.
trainer_summary = (
    trainer_db
    .with_columns(is_ace_spec = pl.col("Rule") == "ACE SPEC")
    .group_by(SUPERTYPE)
    .agg(n_cards=pl.len(), n_ace_spec=pl.col("is_ace_spec").sum())
    .sort("n_cards", descending=True)
)
print(trainer_summary)

# One concrete example of each sub-type:
trainer_db.filter(
    pl.col(SUPERTYPE).is_in(["Item", "Supporter", "Stadium", "Pokémon Tool"])
).group_by(SUPERTYPE).agg(pl.col("Card Name").first().alias("example")).sort(SUPERTYPE)

shape: (4, 3)
┌───────────────────────────────────────────┬─────────┬────────────┐
│ Stage (Pokémon)/Type (Energy and Trainer) ┆ n_cards ┆ n_ace_spec │
│ ---                                       ┆ ---     ┆ ---        │
│ str                                       ┆ u32     ┆ u32        │
╞═══════════════════════════════════════════╪═════════╪════════════╡
│ Item                                      ┆ 77      ┆ 18         │
│ Supporter                                 ┆ 61      ┆ 0          │
│ Pokémon Tool                              ┆ 27      ┆ 6          │
│ Stadium                                   ┆ 26      ┆ 2          │
└───────────────────────────────────────────┴─────────┴────────────┘


Stage (Pokémon)/Type (Energy and Trainer),example
str,str
"""Item""","""Roto-Stick"""
"""Pokémon Tool""","""Team Rocket’s Hypnotizer"""
"""Stadium""","""Community Center"""
"""Supporter""","""Billy & O'Nare"""


### 1d. Energy cards

Energy is attached to Pokémon to pay for attacks (and the retreat cost). The core rule:
**you may attach exactly one Energy from your hand per turn** (barring card effects), so
Energy is the pace-maker of the whole game — attackers that need 3 Energy are a turn slower
than those needing 1.

| Sub-type | What it is | Deck rule |
|---|---|---|
| **Basic Energy** | The 9 plain types (`{G} {R} {W} {L} {P} {F} {D} {M}` + Colorless) | **Unlimited copies** allowed — the *only* exception to the 4-copy rule. |
| **Special Energy** | Energy with extra text (`Legacy`, `Team Rocket's`, `Enriching`, …) | Counts as a normal card: **max 4 copies**, and some are `ACE SPEC` (max 1). |

**Type notation.** Types are written as symbols: `{G}`=Grass, `{R}`=Fire, `{W}`=Water,
`{L}`=Lightning, `{P}`=Psychic, `{F}`=Fighting, `{D}`=Darkness, `{M}`=Metal, `{C}`=Colorless,
plus Dragon and a couple of special markers (`{A}`, `{Team Rocket}`). Attack **costs** use the
same symbols, with a bullet **`●`** standing in for a Colorless (any-type) requirement.

In [30]:
# Basic vs Special energy, and the full Basic Energy list (the unlimited-copies cards).
print(energy_db.group_by(SUPERTYPE).agg(pl.len().alias("n")).sort("n", descending=True))

energy_db.select(["Card ID", "Card Name", SUPERTYPE, "Type", "Rule"])

shape: (2, 2)
┌───────────────────────────────────────────┬─────┐
│ Stage (Pokémon)/Type (Energy and Trainer) ┆ n   │
│ ---                                       ┆ --- │
│ str                                       ┆ u32 │
╞═══════════════════════════════════════════╪═════╡
│ Special Energy                            ┆ 12  │
│ Basic Energy                              ┆ 8   │
└───────────────────────────────────────────┴─────┘


Card ID,Card Name,Stage (Pokémon)/Type (Energy and Trainer),Type,Rule
i64,str,str,str,str
1,"""Basic {G} Energy""","""Basic Energy""","""{G}""","""n/a"""
2,"""Basic {R} Energy""","""Basic Energy""","""{R}""","""n/a"""
3,"""Basic {W} Energy""","""Basic Energy""","""{W}""","""n/a"""
4,"""Basic {L} Energy""","""Basic Energy""","""{L}""","""n/a"""
5,"""Basic {P} Energy""","""Basic Energy""","""{P}""","""n/a"""
6,"""Basic {F} Energy""","""Basic Energy""","""{F}""","""n/a"""
7,"""Basic {D} Energy""","""Basic Energy""","""{D}""","""n/a"""
8,"""Basic {M} Energy""","""Basic Energy""","""{M}""","""n/a"""
9,"""Boomerang Energy""","""Special Energy""","""{C}""","""n/a"""


### 1e. Anatomy of a card's moves (the multi-row structure)

Recall each **row is one move** (attack or ability). A Pokémon with a Tera ability and two
attacks is three rows sharing one `Card ID`. The move columns:

- **`Move Name`** — e.g. `[Tera]` denotes an ability rather than an attack.
- **`Cost`** — Energy needed, as type symbols + `●` for Colorless (e.g. `{G}{L}{F}`, `{M}{M}●●`).
- **`Damage`** — base damage. Watch for modifiers: **`×`** means variable ("20× the number
  of …"), and a value can be blank for pure-effect attacks.
- **`Effect Explanation`** — the rules text.

Plus the card-level battle stats that matter for combat math:
- **`Weakness`** — takes **double** damage from that type.
- **`Resistance (Type)`** — takes **−30** from that type.
- **`Retreat`** — Colorless Energy that must be discarded to switch this Pokémon out of the
  Active spot.

Below: a full multi-attack card, then the raw damage tokens so you can see the modifiers we'll
have to parse for the agent.

In [31]:
# A card whose moves span multiple rows (ability + two attacks):
example_id = (
    df.group_by("Card ID").len().sort("len", descending=True).row(0)[0]
)
print(f"Example multi-move card (Card ID {example_id}):")
print(df.filter(pl.col("Card ID") == example_id)
        .select(["Card Name", "Move Name", "Cost", "Damage", "Effect Explanation"]))

# The damage tokens that carry modifiers we must handle when we num-ify damage for the agent:
dmg = df.select("Damage").filter(pl.col("Damage") != "n/a").to_series()
print("\nDamage tokens containing a modifier (× variable, +, -):")
print(sorted(d for d in dmg.unique() if any(c in d for c in "×+-")))

Example multi-move card (Card ID 161):
shape: (3, 5)
┌───────────────┬─────────────┬───────────┬────────┬───────────────────────────────────────────────┐
│ Card Name     ┆ Move Name   ┆ Cost      ┆ Damage ┆ Effect Explanation                            │
│ ---           ┆ ---         ┆ ---       ┆ ---    ┆ ---                                           │
│ str           ┆ str         ┆ str       ┆ str    ┆ str                                           │
╞═══════════════╪═════════════╪═══════════╪════════╪═══════════════════════════════════════════════╡
│ Galvantula ex ┆ [Tera]      ┆ n/a       ┆ n/a    ┆ As long as this Pokémon is on your Bench,     │
│               ┆             ┆           ┆        ┆ prevent all damage…                           │
│ Galvantula ex ┆ Charged Web ┆ {L}●      ┆ 110    ┆ If your opponent’s Active Pokémon is a        │
│               ┆             ┆           ┆        ┆ Pokémon {ex} or Pokém…                        │
│ Galvantula ex ┆ Fulgurite   ┆ {G}{L}

## 2. Preparing the data for the RL agent

The exploration above is for humans. An RL agent needs **numeric, fixed-width, id-aligned**
tables. The agent has three jobs, and each wants the data in a specific shape:

1. **Build a deck** — pick 60 card IDs that are *legal* and *synergistic*. Needs a **per-card
   feature matrix** (one row per card ID) plus a **legality checker** so it never proposes an
   illegal deck.
2. **Play games** — at each decision point the engine hands the agent an `Observation` full of
   `Card ID`s (hand, board, prizes…). To turn those into a state vector it needs an **O(1)
   id → feature lookup** — the same per-card matrix, indexable by id.
3. **Reevaluate & refine** — after many games it correlates *which cards were in the deck* with
   *wins*. That's a join between deck composition (feature matrix) and game outcomes.

**Design choice:** we build the features from the **engine** (`all_card_data()` /
`all_attack()`), not the CSV. The engine is what the agent sees at runtime — its enums,
HP, retreat, weakness, evolution and attack data are already typed and are guaranteed
consistent with in-game behavior. We attach the CSV's readable name/effect text only where it
helps debugging.

In [32]:
from dataclasses import asdict

from cg.api import all_card_data, all_attack, CardType, EnergyType

# --- Engine card table (the ground truth the agent sees at runtime) ---
card_data = all_card_data()          # list[CardData], one per Card ID
attack_data = all_attack()           # list[Attack], referenced by CardData.attacks

print(f"engine cards  : {len(card_data)}")
print(f"engine attacks: {len(attack_data)}")

# Sanity check: the engine and the CSV describe the same universe of cards.
engine_ids = {c.cardId for c in card_data}
csv_ids = set(cards["Card ID"].to_list())
assert engine_ids == csv_ids, "engine / CSV card-id mismatch!"
print(f"engine and CSV agree on all {len(engine_ids)} card ids ✔")

# Readable enum labels we'll reuse for column names.
ENERGY_NAMES = {e.value: e.name.title() for e in EnergyType}   # 0->Colorless, 1->Grass, ...
CARDTYPE_NAMES = {t.value: t.name for t in CardType}
print("\nenergy types:", ENERGY_NAMES)
print("card types  :", CARDTYPE_NAMES)

engine cards  : 1267
engine attacks: 1556
engine and CSV agree on all 1267 card ids ✔

energy types: {0: 'Colorless', 1: 'Grass', 2: 'Fire', 3: 'Water', 4: 'Lightning', 5: 'Psychic', 6: 'Fighting', 7: 'Darkness', 8: 'Metal', 9: 'Dragon', 10: 'Rainbow', 11: 'Team_Rocket'}
card types  : {0: 'POKEMON', 1: 'ITEM', 2: 'TOOL', 3: 'SUPPORTER', 4: 'STADIUM', 5: 'BASIC_ENERGY', 6: 'SPECIAL_ENERGY'}


### 2a. Attack table — `attacks_ft`

One row per attack, keyed by `attackId`. The engine already gives `energies` as a list of
`EnergyType` ints, so we **vector-encode the cost**: a `cost_<Type>` count column per energy
type plus `cost_total`. Damage is a clean int (`0` for pure-effect attacks); we add
`is_variable` for attacks whose damage scales ("×"/"for each …"), since the flat number
understates them. This is what the agent uses to reason about "can I afford this attack and
how hard does it hit."

In [33]:
from collections import Counter

# Only the "real" energy types actually used in costs (skip Rainbow/Team-Rocket combo markers,
# which don't appear as attack costs). Colorless(0) is the "any energy" / ● requirement.
COST_TYPES = [EnergyType.COLORLESS, EnergyType.GRASS, EnergyType.FIRE, EnergyType.WATER,
              EnergyType.LIGHTNING, EnergyType.PSYCHIC, EnergyType.FIGHTING,
              EnergyType.DARKNESS, EnergyType.METAL, EnergyType.DRAGON]

def attack_row(a) -> dict:
    counts = Counter(a.energies)                       # e.g. Counter({8: 2}) -> two Metal
    row = {"attackId": a.attackId, "attack_name": a.name,
           "damage": a.damage, "cost_total": len(a.energies)}
    for e in COST_TYPES:
        row[f"cost_{ENERGY_NAMES[e.value]}"] = counts.get(e.value, 0)
    text = (a.text or "").lower()
    row["is_variable"] = ("×" in a.text) or ("for each" in text) or ("damage for each" in text)
    return row

attacks_ft = pl.DataFrame([attack_row(a) for a in attack_data])
print(attacks_ft.shape, "->", attacks_ft.columns)
attacks_ft.head(6)

(1556, 15) -> ['attackId', 'attack_name', 'damage', 'cost_total', 'cost_Colorless', 'cost_Grass', 'cost_Fire', 'cost_Water', 'cost_Lightning', 'cost_Psychic', 'cost_Fighting', 'cost_Darkness', 'cost_Metal', 'cost_Dragon', 'is_variable']


attackId,attack_name,damage,cost_total,cost_Colorless,cost_Grass,cost_Fire,cost_Water,cost_Lightning,cost_Psychic,cost_Fighting,cost_Darkness,cost_Metal,cost_Dragon,is_variable
i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,bool
1,"""Nab ’n’ Dash""",0,1,1,0,0,0,0,0,0,0,0,0,false
2,"""High Jump Kick""",100,3,2,0,0,0,0,0,0,1,0,0,false
3,"""Push Down""",10,1,0,0,0,0,0,0,1,0,0,0,false
4,"""Ram""",60,2,0,0,0,0,0,0,2,0,0,0,false
5,"""Super Sandstorm""",150,3,1,0,0,0,0,0,2,0,0,0,false
6,"""Comet Punch""",0,2,2,0,0,0,0,0,0,0,0,0,true


### 2b. Per-card feature matrix — `cards_ft`

The workhorse table: **one row per `Card ID`**, all numeric, ready to be indexed by id during
play or stacked into a deck embedding during deck-building. It carries:

- **Identity / type:** `card_type` (int enum) + one-hot `is_pokemon / is_item / is_supporter /
  is_stadium / is_tool / is_basic_energy / is_special_energy`.
- **Battle stats:** `hp`, `retreat_cost`, one-hot **energy type**, and `weakness_* / resistance_*`
  type ids (−1 when none).
- **Evolution:** `is_basic / is_stage1 / is_stage2`, `evolves_from_id` (−1 if none).
- **Rule overlays:** `is_ex / is_mega_ex / is_tera / is_ace_spec`, and `prizes_on_ko`.
- **Aggregated attack power** (joined from `attacks_ft`): `n_attacks`, `max_damage`,
  `min_attack_cost`, `has_variable_attack` — a compact summary of offensive potential without
  the agent having to walk the attack list every time.

Everything is keyed so `cards_ft.filter(pl.col("card_id") == some_id)` is the id→features
lookup the play-time state encoder needs.

In [34]:
# Lookups: attackId -> its stats, and card name -> id (to resolve evolvesFrom names to ids).
atk_by_id = {a.attackId: a for a in attack_data}
name_to_id = {c.name: c.cardId for c in card_data}       # last-wins; names are ~unique enough here

def energy_or(v, default=-1):
    """EnergyType|None -> int id (or -1 when the card has no weakness/resistance)."""
    return int(v) if v is not None else default

def card_row(c) -> dict:
    is_pokemon = c.cardType == CardType.POKEMON
    row = {
        "card_id": c.cardId,
        "name": c.name,
        "card_type": int(c.cardType),
        # one-hot super-type
        "is_pokemon":        c.cardType == CardType.POKEMON,
        "is_item":           c.cardType == CardType.ITEM,
        "is_supporter":      c.cardType == CardType.SUPPORTER,
        "is_stadium":        c.cardType == CardType.STADIUM,
        "is_tool":           c.cardType == CardType.TOOL,
        "is_basic_energy":   c.cardType == CardType.BASIC_ENERGY,
        "is_special_energy": c.cardType == CardType.SPECIAL_ENERGY,
        # battle stats
        "hp": c.hp,
        "retreat_cost": c.retreatCost,
        "energy_type_id": int(c.energyType),
        "weakness_id":   energy_or(c.weakness),
        "resistance_id": energy_or(c.resistance),
        # evolution
        "is_basic":  c.basic,
        "is_stage1": c.stage1,
        "is_stage2": c.stage2,
        "evolves_from_id": name_to_id.get(c.evolvesFrom, -1) if c.evolvesFrom else -1,
        # rule overlays + prize value
        "is_ex":        c.ex,
        "is_mega_ex":   c.megaEx,
        "is_tera":      c.tera,
        "is_ace_spec":  c.aceSpec,
        "prizes_on_ko": (3 if c.megaEx else 2 if c.ex else 1) if is_pokemon else 0,
    }
    # one-hot energy type (the 10 real types)
    for e in COST_TYPES:
        row[f"type_{ENERGY_NAMES[e.value]}"] = (is_pokemon or c.cardType == CardType.BASIC_ENERGY) \
                                               and c.energyType == e
    # aggregated attack power
    atks = [atk_by_id[i] for i in c.attacks if i in atk_by_id]
    row["n_attacks"]           = len(atks)
    row["max_damage"]          = max((a.damage for a in atks), default=0)
    row["min_attack_cost"]     = min((len(a.energies) for a in atks), default=0)
    row["has_variable_attack"] = any("×" in a.text or "for each" in (a.text or "").lower() for a in atks)
    return row

cards_ft = pl.DataFrame([card_row(c) for c in card_data]).sort("card_id")
print(f"cards_ft: {cards_ft.shape[0]} cards x {cards_ft.shape[1]} features")
print(cards_ft.columns)
cards_ft.head(8)

cards_ft: 1267 cards x 38 features
['card_id', 'name', 'card_type', 'is_pokemon', 'is_item', 'is_supporter', 'is_stadium', 'is_tool', 'is_basic_energy', 'is_special_energy', 'hp', 'retreat_cost', 'energy_type_id', 'weakness_id', 'resistance_id', 'is_basic', 'is_stage1', 'is_stage2', 'evolves_from_id', 'is_ex', 'is_mega_ex', 'is_tera', 'is_ace_spec', 'prizes_on_ko', 'type_Colorless', 'type_Grass', 'type_Fire', 'type_Water', 'type_Lightning', 'type_Psychic', 'type_Fighting', 'type_Darkness', 'type_Metal', 'type_Dragon', 'n_attacks', 'max_damage', 'min_attack_cost', 'has_variable_attack']


card_id,name,card_type,is_pokemon,is_item,is_supporter,is_stadium,is_tool,is_basic_energy,is_special_energy,hp,retreat_cost,energy_type_id,weakness_id,resistance_id,is_basic,is_stage1,is_stage2,evolves_from_id,is_ex,is_mega_ex,is_tera,is_ace_spec,prizes_on_ko,type_Colorless,type_Grass,type_Fire,type_Water,type_Lightning,type_Psychic,type_Fighting,type_Darkness,type_Metal,type_Dragon,n_attacks,max_damage,min_attack_cost,has_variable_attack
i64,str,i64,bool,bool,bool,bool,bool,bool,bool,i64,i64,i64,i64,i64,bool,bool,bool,i64,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64,i64,i64,bool
1,"""Basic {G} Energy""",5,false,false,false,false,false,true,false,0,0,1,-1,-1,false,false,false,-1,false,false,false,false,0,false,true,false,false,false,false,false,false,false,false,0,0,0,false
2,"""Basic {R} Energy""",5,false,false,false,false,false,true,false,0,0,2,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,true,false,false,false,false,false,false,false,0,0,0,false
3,"""Basic {W} Energy""",5,false,false,false,false,false,true,false,0,0,3,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,true,false,false,false,false,false,false,0,0,0,false
4,"""Basic {L} Energy""",5,false,false,false,false,false,true,false,0,0,4,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,false,true,false,false,false,false,false,0,0,0,false
5,"""Basic {P} Energy""",5,false,false,false,false,false,true,false,0,0,5,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,false,false,true,false,false,false,false,0,0,0,false
6,"""Basic {F} Energy""",5,false,false,false,false,false,true,false,0,0,6,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,false,false,false,true,false,false,false,0,0,0,false
7,"""Basic {D} Energy""",5,false,false,false,false,false,true,false,0,0,7,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,false,false,false,false,true,false,false,0,0,0,false
8,"""Basic {M} Energy""",5,false,false,false,false,false,true,false,0,0,8,-1,-1,false,false,false,-1,false,false,false,false,0,false,false,false,false,false,false,false,false,true,false,0,0,0,false


### 2c. Deck-building legality — `validate_deck()`

For **job 1 (build a deck)** the agent must only ever emit *legal* 60-card decks. Rather than
hope it learns the rules, we hand it a checker it can call (as a hard constraint / action mask
during search, or as a reward penalty during training). The Pokémon TCG deck rules:

1. **Exactly 60 cards.**
2. **At most 4 copies** of any card **by name** — *except* **Basic Energy**, which is unlimited.
3. **At least one Basic Pokémon** (otherwise you can't legally start).
4. **At most one `ACE SPEC`** card in the whole deck.

`validate_deck(ids)` returns `(is_legal, reasons)` using `cards_ft` for the per-card facts. We
test it on the repo's `deck.csv` (the 60-card list the sample agent submits).

In [35]:
# Fast per-id lookups pulled once from the feature matrix.
_ft = {r["card_id"]: r for r in cards_ft.iter_rows(named=True)}

DECK_SIZE = 60
MAX_COPIES = 4

def validate_deck(ids: list[int]) -> tuple[bool, list[str]]:
    """Check a 60-card deck (list of Card IDs) against the deck-construction rules."""
    reasons = []

    if len(ids) != DECK_SIZE:
        reasons.append(f"deck has {len(ids)} cards, must be exactly {DECK_SIZE}")

    unknown = [i for i in ids if i not in _ft]
    if unknown:
        reasons.append(f"unknown card ids: {sorted(set(unknown))}")

    known = [i for i in ids if i in _ft]

    # Rule 2: <=4 copies by NAME, Basic Energy exempt.
    copies_by_name: Counter[str] = Counter()
    for i in known:
        if not _ft[i]["is_basic_energy"]:
            copies_by_name[_ft[i]["name"]] += 1
    over = {n: c for n, c in copies_by_name.items() if c > MAX_COPIES}
    if over:
        reasons.append(f">4 copies of: {over}")

    # Rule 3: at least one Basic Pokemon.
    if not any(_ft[i]["is_basic"] and _ft[i]["is_pokemon"] for i in known):
        reasons.append("no Basic Pokémon (cannot legally start a game)")

    # Rule 4: at most one ACE SPEC.
    n_ace = sum(_ft[i]["is_ace_spec"] for i in known)
    if n_ace > 1:
        reasons.append(f"{n_ace} ACE SPEC cards (max 1)")

    return (len(reasons) == 0, reasons)


# Test on the repo's submitted deck.
deck_ids = [int(x) for x in Path("../deck.csv").read_text().split() if x.strip()]
legal, why = validate_deck(deck_ids)
print(f"deck.csv -> {len(deck_ids)} cards | legal={legal}")
for r in why:
    print("  -", r)

# A couple of deliberately-illegal decks to show the checker bites:
print("\n5-of-a-card :", validate_deck([deck_ids[0]] * 5 + deck_ids[5:])[1])
print("59 cards    :", validate_deck(deck_ids[:59])[1])

deck.csv -> 60 cards | legal=True

5-of-a-card : [">4 copies of: {'Maximum Belt': 5}", '5 ACE SPEC cards (max 1)']
59 cards    : ['deck has 59 cards, must be exactly 60']


### 2d. Persist the artifacts

We write the two tables to `data/` so the training/serving code doesn't have to reload the
engine and re-derive features every run. Parquet keeps dtypes; a CSV copy is handy for eyeballing.

**How the agent consumes these**

| Job | Uses |
|---|---|
| **Build deck** | `cards_ft` as the candidate pool + `validate_deck()` as a hard legality mask. |
| **Play games** | `cards_ft` as an `id → feature-vector` table to encode each `Observation` into state; `attacks_ft` to evaluate attack affordability/damage. |
| **Reevaluate** | Join per-game win/loss outcomes against deck composition (`cards_ft`) to learn which cards/archetypes correlate with winning, then feed that back into deck-building. |

That closes the loop: build → play → reevaluate → rebuild, and (the plan) start winning bigly. 🏆

In [36]:
out_dir = Path("../data")

cards_ft.write_parquet(out_dir / "cards_features.parquet")
cards_ft.write_csv(out_dir / "cards_features.csv")
attacks_ft.write_parquet(out_dir / "attacks_features.parquet")

print("wrote:")
for p in ["cards_features.parquet", "cards_features.csv", "attacks_features.parquet"]:
    f = out_dir / p
    print(f"  {f}  ({f.stat().st_size / 1024:.0f} KB)")

# Reload smoke-test: this is exactly the id->features lookup the play-time encoder will do.
_check = pl.read_parquet(out_dir / "cards_features.parquet")
assert _check.shape == cards_ft.shape
print(f"\nreload OK — {_check.height} cards x {_check.width} features")

wrote:
  data\cards_features.parquet  (30 KB)
  data\cards_features.csv  (235 KB)
  data\attacks_features.parquet  (23 KB)

reload OK — 1267 cards x 38 features
